# Land Cover Classification with Prithvi-EO-2.0
**PyGeoVision v2.0 — Notebook 03**

## Real-World Problem
Environmental agencies need to produce land-cover maps for climate reporting.
Prithvi-EO-2.0 (600M params, pre-trained on 10 years of global HLS data) enables
state-of-the-art classification with minimal fine-tuning.

## Learning Objectives
1. Download multi-spectral Sentinel-2 (6 bands for Prithvi HLS format)
2. Apply the correct band mapping (S2 → HLS order)
3. Run Prithvi-EO-2.0 land cover classification
4. Visualise 9-class results with area statistics
5. Fine-tune Prithvi on custom data

## Prerequisites
```bash
pip install "pygeovision[geo,foundation]" transformers huggingface_hub
```

In [ ]:
import pygeovision as pgv
import numpy as np, rasterio, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/03_land_cover_prithvi")
BASE_DIR.mkdir(parents=True, exist_ok=True)

BBOX      = (-8.90, 38.60, -8.50, 38.90)   # Lisbon region, Portugal
DATE_RANGE = ("2024-06-01", "2024-09-30")

client = pgv.PyGeoVision()
print(client)
print(f"\nStudy area : Lisbon region, Portugal")
print(f"BBOX       : {BBOX}")

## Step 1: Download 6-Band Sentinel-2 (Prithvi Format)

In [ ]:
# Prithvi requires EXACTLY these 6 bands in HLS order:
# Blue(B02), Green(B03), Red(B04), NIR(B08), SWIR1(B11), SWIR2(B12)
PRITHVI_S2_BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]

results = client.search(
    bbox=BBOX, date_range=DATE_RANGE,
    providers=["planetary_computer"], cloud_cover_max=5,
    sort_by="cloud_cover", limit=3,
)
print(f"Found {len(results)} scenes")
scene_path = None
if results:
    dl = client.download(
        results[:1], output_dir=BASE_DIR/"scenes",
        bands=PRITHVI_S2_BANDS,
        post_process=["reproject:EPSG:32629","cog"],
    )
    if dl and dl[0].success:
        scene_path = dl[0].path
        print(f"Downloaded: {Path(scene_path).name}")

## Step 2: Band Mapping (Critical!)

In [ ]:
from pygeovision.models.foundation.prithvi import map_bands, normalise_hls

print("CRITICAL: Prithvi expects HLS (Harmonized Landsat Sentinel-2) band order")
print()
print(f"{'Position':<10} {'HLS Band':<12} {'Sentinel-2':<14} {'Landsat':<10} {'Wavelength'}")
print("─" * 60)
band_info = [
    ("0", "Blue",  "B02", "B2", "~490 nm"),
    ("1", "Green", "B03", "B3", "~560 nm"),
    ("2", "Red",   "B04", "B4", "~665 nm"),
    ("3", "NIR",   "B08", "B5", "~842 nm"),
    ("4", "SWIR1", "B11", "B6", "~1610 nm"),
    ("5", "SWIR2", "B12", "B7", "~2190 nm"),
]
for pos, hls, s2, ls, wl in band_info:
    print(f"  {pos:<8}  {hls:<12} {s2:<14} {ls:<10} {wl}")

print()
if scene_path and Path(scene_path).exists():
    with rasterio.open(scene_path) as src:
        data = src.read().astype(np.float32)
    print(f"Input shape : {data.shape}  (bands, height, width)")
    data_hls  = map_bands(data, source="sentinel2", n_prithvi_bands=6)
    data_norm = normalise_hls(data_hls)
    print(f"After mapping: {data_hls.shape}")
    print(f"After normalise: range=[{data_norm.min():.4f}, {data_norm.max():.4f}]")
    print(f"  HLS scale factor: 10000.0 (reflectance * 10000 = int)")
else:
    print("No scene — using demo array")
    np.random.seed(42)
    data = np.random.randint(100, 3000, (6, 512, 512), dtype=np.uint16).astype(np.float32)
    data_hls  = map_bands(data, source="sentinel2", n_prithvi_bands=6)
    data_norm = normalise_hls(data_hls)
    print(f"Demo shape: {data_norm.shape}")

## Step 3: Load Prithvi-EO-2.0 and Classify

In [ ]:
from pygeovision.models.foundation.prithvi import Prithvi, PrithviTasks

# Model specification
print("Prithvi-EO Foundation Models:")
print()
models = {
    "prithvi_eo_1_0": {"params": "100M", "coverage": "USA", "temporal": "3 frames"},
    "prithvi_eo_2_0": {"params": "600M", "coverage": "Global", "temporal": "4 frames"},
}
for name, info in models.items():
    print(f"  {name}")
    for k, v in info.items():
        print(f"    {k:<12}: {v}")
print()

# Land cover class definitions (ESA WorldCover compatible)
CLASSES = ["Water", "Trees", "Grassland", "Flooded veg", "Crops",
           "Shrub", "Built-up", "Bare/sparse", "Snow/Ice"]
COLORS  = ["#1A5276","#196F3D","#7DCEA0","#1ABC9C","#F1C40F",
            "#E67E22","#E74C3C","#D5D8DC","#AED6F1"]

# Task prediction
tasks = PrithviTasks("prithvi_eo_2_0")
print("Running Prithvi-EO-2.0 land cover classification...")

if scene_path and Path(scene_path).exists():
    result   = tasks.land_cover(scene_path, source="sentinel2",
                                  output_path=str(BASE_DIR/"land_cover.tif"))
    pred     = result["prediction"]
    class_pct= result["class_pct"]
else:
    print("(Demo mode — generating simulated land cover)")
    np.random.seed(99)
    H, W    = 256, 256
    pred    = np.zeros((H,W), dtype=np.uint8)
    # Lisbon region: mostly built-up, trees, crops
    probs   = [0.06, 0.28, 0.10, 0.03, 0.22, 0.07, 0.16, 0.06, 0.02]
    pred    = np.random.choice(len(CLASSES), size=(H,W), p=probs).astype(np.uint8)
    class_pct = {i: float((pred==i).mean()*100) for i in range(len(CLASSES))}

print("\nLand Cover Results:")
print(f"{'Class':<18} {'Coverage':>10} {'Bar'}")
print("─" * 50)
for i, name in enumerate(CLASSES):
    pct = class_pct.get(i, 0)
    bar = "█" * int(pct / 2)
    print(f"  {name:<16} {pct:>8.1f}%  {bar}")

## Step 4: Visualise Land Cover Map

In [ ]:
cmap = mcolors.ListedColormap(COLORS)
norm = mcolors.BoundaryNorm(range(len(CLASSES)+1), cmap.N)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im = axes[0].imshow(pred, cmap=cmap, norm=norm)
axes[0].set_title("Land Cover Map\nPrithvi-EO-2.0 (9 classes)", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Colourbar with class names
cbar = plt.colorbar(im, ax=axes[0], fraction=0.03, pad=0.02)
cbar.set_ticks([i + 0.5 for i in range(len(CLASSES))])
cbar.set_ticklabels(CLASSES, fontsize=8)

# Pie chart (non-zero classes)
pcts   = [class_pct.get(i,0) for i in range(len(CLASSES))]
nonzero = [(n,p,c) for n,p,c in zip(CLASSES,pcts,COLORS) if p > 0.5]
if nonzero:
    labels, values, colors = zip(*nonzero)
    axes[1].pie(values, labels=labels, colors=colors, autopct='%1.1f%%',
                 startangle=90, textprops={'fontsize':9})
axes[1].set_title("Land Cover Distribution\nLisbon Region, Portugal", fontsize=12, fontweight='bold')

plt.suptitle("Land Cover Classification — Prithvi-EO-2.0 (600M)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR/"land_cover_map.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {BASE_DIR/'land_cover_map.png'}")

## Step 5: Area Statistics

In [ ]:
PIXEL_SIZE_M = 10   # Sentinel-2 resolution
PIXEL_HA     = (PIXEL_SIZE_M ** 2) / 10000

print("=== Land Cover Area Statistics ===")
print()
print(f"{'Class':<18} {'Area (ha)':>10} {'Coverage':>10}")
print("─" * 42)
total_pix = pred.size
for i, name in enumerate(CLASSES):
    pct  = class_pct.get(i, 0)
    npix = int(total_pix * pct / 100)
    ha   = npix * PIXEL_HA
    if pct > 0.1:
        print(f"  {name:<16} {ha:>9,.0f}  {pct:>9.1f}%")

total_ha = total_pix * PIXEL_HA
print(f"  {'Total':<16} {total_ha:>9,.0f}  {'100.0':>9}%")
print()

# Class change sensitivity: ±1 pixel threshold
print("Classification confidence notes:")
print("  Prithvi-EO-2.0 typical accuracy:")
print("  - Water:     97% F1 (spectrally distinctive)")
print("  - Trees:     91% F1 (dense canopy)")
print("  - Crops:     84% F1 (seasonal variability)")
print("  - Built-up:  88% F1 (spectral mixing at 10m)")

## Step 6: Fine-Tuning on Custom Data

In [ ]:
from pygeovision.models.foundation.prithvi import finetune_prithvi

print("Fine-tuning Prithvi-EO-2.0 for custom land cover:")
print()
print("Configuration:")
cfg = {
    "model_name"     : "prithvi_eo_2_0",
    "task"           : "land_cover",
    "num_classes"    : 9,
    "epochs"         : 50,
    "learning_rate"  : 5e-5,      # Paper recommendation
    "weight_decay"   : 0.01,
    "batch_size"     : 8,         # Memory-limited for 600M params
    "mixed_precision": True,      # BF16 for modern GPUs (A100, H100)
    "warmup_epochs"  : 5,
    "freeze_encoder" : False,     # Full fine-tuning recommended
}
for k, v in cfg.items():
    print(f"  {k:<20}: {v}")
print()
print("To run (requires training data):")
print("  result = finetune_prithvi(**cfg, output_dir='./checkpoints/prithvi_lc/')")
print("  model  = result['model']")
print("  # Best practice: freeze encoder for first 5 epochs, then unfreeze")

## Step 7: Multi-Temporal Analysis with Prithvi

In [ ]:
from pygeovision.models.foundation.prithvi import PrithviMultiTemporal

mt = PrithviMultiTemporal("prithvi_eo_2_0")
print("Prithvi Multi-Temporal capabilities:")
print()
print("  mt.process_time_series(images, dates=dates)")
print("    -> extracts temporal features across 4 frames simultaneously")
print()
print("  mt.detect_change(before.tif, after.tif)")
print("    -> uses temporal attention to find structural changes")
print()
print("  mt.monitor_trend(monthly_images)")
print("    -> linear fit on temporal features -> trend direction + slope")
print()
print("  mt.predict_seasonal(images, dates=dates)")
print("    -> seasonal amplitude, peak/trough months")
print()
print("Advantage over single-image models:")
print("  Prithvi sees CONTEXT across time -> distinguishes")
print("  seasonal change (crops) from permanent change (deforestation)")

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 03 — Land Cover Classification with Prithvi")
print("=" * 60)
print()
print("Model    : Prithvi-EO-2.0 (600M parameters)")
print("Pre-train: HLS Global 10-year archive")
print("Classes  : 9 (ESA WorldCover compatible)")
print("Input    : 6-band multispectral (B,G,R,NIR,SWIR1,SWIR2)")
print()
print("Key insight:")
print("  The band mapping step is CRITICAL.")
print("  Wrong order -> silent 10-20 mIoU degradation.")
print("  Always use: map_bands(data, source='sentinel2')")
print()
print("Next: 04_change_detection_temporal_analysis.ipynb")